# The Brachistochrone — the Fastest Slide as a Convex Problem

**Continuous Optimization (MasterMath) — Lecture 1 demo.**

Johann Bernoulli's challenge (1696): along which curve does a bead slide under gravity, without friction, from the origin to a point $L$ to the right and $H$ below, in the *least time* [2]? By conservation of energy the speed at depth $y$ is $\sqrt{2gy}$, whatever the curve. Parametrise a curve as $z(s) = (x(s), y(s))$, $s \in [0,1]$; its travel time is arc length over speed:

$$T[z] = \int_0^1 \frac{\|z'(s)\|}{\sqrt{2g\,y(s)}}\, ds.$$

**Reparametrisation freedom.** Many different $z$ describe the *same* physical curve: composing with any increasing bijection of $[0,1]$ changes the function $z$ but not the value $T[z]$ (substitute variables in the integral). The curve is *overparametrised* — and this freedom is ours to spend.

**Spending it well.** Write $m(s) := \|z'(s)\|/\sqrt{2g\,y(s)}$ for the integrand. For any given curve we can make $m$ **constant**: let $\tau(s) := \int_0^s m(u)\,du$ be the time elapsed up to $s$ (so $\tau(1) = T$), and reparametrise by normalised elapsed time, $\bar z(s) := z(\tau^{-1}(sT))$. The chain rule gives $\bar m(s) \equiv T$: the bead's clock ticks evenly in the new parameter.

**Why this helps.** By Cauchy–Schwarz,

$$E[z] := \int_0^1 m(s)^2\, ds \;\ge\; \left(\int_0^1 m(s)\, ds\right)^{\!2} = T[z]^2,$$

with equality precisely when $m$ is constant. Since every curve admits a constant-$m$ parametrisation, minimising $E$ over all parametrised curves is the *same problem* as minimising $T^2$ — but $E$, unlike $T$, is **convex**: its integrand $\|p\|^2/(2gy)$ is the *perspective function*, jointly convex in $(p, y)$ [1], and $z \mapsto (z(s), z'(s))$ is linear. Boundary conditions are linear too.

Discretised on $s_k = k/N$, each segment contributes $N\,\|z_{k+1}-z_k\|^2 / (2g\,\bar y_k)$ with $\bar y_k$ the segment's mean depth — a `quad_over_lin` term — so the whole problem is a small second-order-cone program. And since the optimiser has constant $m$, the recovered $s$ **is** time (divided by $T$).

In [ ]:
# Colab does not ship CVXPY by default; install it (skipped if already present).
try:
    import cvxpy  # noqa: F401
except ImportError:
    %pip install -q cvxpy

In [ ]:
import numpy as np
import cvxpy as cp

g = 9.81         # gravity (m/s^2)
L, H = 2.5, 1.0  # end point: L to the right, H straight down

# The curve z_k = (x_k, y_k) at s_k = k/N, completely free.
N = 300
Z = cp.Variable((N + 1, 2))
dZ = cp.diff(Z, axis=0)
y_seg = (Z[:-1, 1] + Z[1:, 1]) / 2      # mean depth of each segment

E = N * cp.sum(cp.vstack([cp.quad_over_lin(dZ[k], y_seg[k])
                          for k in range(N)])) / (2 * g)

problem = cp.Problem(cp.Minimize(E), [Z[0] == [0, 0], Z[N] == [L, H]])
problem.solve()

print(f"Status: {problem.status}")
print(f"Travel time = sqrt(E) : {np.sqrt(problem.value):.6f} s")
print(f"Deepest point reached : {Z.value[:, 1].max():.4f}  (destination: {H})")

**The classical answer** (Bernoulli, Newton, Leibniz, l'Hôpital, and Jacob Bernoulli all solved it) is a **cycloid**: $x = r(\theta - \sin\theta)$, $y = r(1 - \cos\theta)$, the path traced by a point on a rolling wheel, with travel time $\Theta\sqrt{r/g}$ at the angle $\Theta$ where it reaches $(L, H)$. For our end point ($L/H > \pi/2$) the cycloid even *dips below its destination* and climbs back up — let us check that the solver rediscovers all of this.

In [ ]:
from scipy.optimize import brentq

# Angle at which the cycloid through the origin reaches (L, H).
theta = brentq(lambda t: (t - np.sin(t)) / (1 - np.cos(t)) - L / H,
               1e-3, 2 * np.pi - 1e-3)
r = H / (1 - np.cos(theta))
print(f"Travel time (Bernoulli's cycloid) : {theta * np.sqrt(r / g):.6f} s")
print(f"Deepest point of the cycloid      : {r * (1 - np.cos(min(theta, np.pi))):.4f}")

In [ ]:
import matplotlib.pyplot as plt

t_c = np.linspace(0, theta, 400)

plt.figure(figsize=(8, 4.5))
plt.plot(Z.value[:, 0], Z.value[:, 1], lw=4, alpha=0.5, label="convex optimization")
plt.plot(r * (t_c - np.sin(t_c)), r * (1 - np.cos(t_c)), "k--", lw=1.5,
         label="Bernoulli's cycloid")
plt.axhline(H, color="gray", lw=0.5)
plt.scatter([0, L], [0, H], color="k", zorder=3)
plt.gca().invert_yaxis()               # depth increases downward
plt.gca().set_aspect("equal")
plt.xlabel("horizontal position $x$")
plt.ylabel("depth $y$")
plt.title("The fastest slide is a cycloid — dipping below its destination")
plt.legend()
plt.show()

## Beyond the reach of pen and paper: an obstacle

The cycloid was found analytically — but now suppose a **floor** blocks the dip: the slide may not go deeper than $y \le d$, with $d$ above the cycloid's lowest point. The optimal curve becomes cycloid arc – straight run along the floor – cycloid arc, and no elementary formula exists for it any more. The convex problem hardly notices: the floor is one *linear* constraint per grid point, $y_k \le d$.

(The curve's feasible set must stay convex — a floor, ceiling or any half-plane qualifies; keeping out of a disc would not.)

In [ ]:
d = 1.05                                # floor depth, above the free dip

Zf = cp.Variable((N + 1, 2))
dZf = cp.diff(Zf, axis=0)
yf_seg = (Zf[:-1, 1] + Zf[1:, 1]) / 2
Ef = N * cp.sum(cp.vstack([cp.quad_over_lin(dZf[k], yf_seg[k])
                           for k in range(N)])) / (2 * g)

problem_f = cp.Problem(cp.Minimize(Ef),
                       [Zf[0] == [0, 0], Zf[N] == [L, H], Zf[:, 1] <= d])
problem_f.solve()

print(f"Status: {problem_f.status}")
print(f"Travel time with the floor    : {np.sqrt(problem_f.value):.6f} s")
print(f"Travel time without the floor : {np.sqrt(problem.value):.6f} s")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(Z.value[:, 0], Z.value[:, 1], lw=1.5, ls="--", label="free optimum (blocked)")
plt.plot(Zf.value[:, 0], Zf.value[:, 1], lw=4, alpha=0.6, color="tab:green",
         label="fastest slide above the floor")
plt.axhspan(d, 1.25, color="tab:gray", alpha=0.4, label="floor")
plt.axhline(H, color="gray", lw=0.5)
plt.scatter([0, L], [0, H], color="k", zorder=3)
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.ylim(1.25, -0.05)
plt.xlabel("horizontal position $x$")
plt.ylabel("depth $y$")
plt.title("No analytical solution — no problem")
plt.legend()
plt.show()

**References**

1. S. Boyd and L. Vandenberghe, *Convex Optimization*, Cambridge University Press, 2004, §3.2.6 — the perspective function and its convexity (CVXPY's `quad_over_lin`).
2. H. J. Sussmann and J. C. Willems, "300 years of optimal control: from the brachystochrone to the maximum principle", *IEEE Control Systems Magazine* 17(3):32–44, 1997 — the problem's history and Bernoulli's own solution.